In [10]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import re
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

In [11]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "data_registry" / "dataset_candidate_registry.csv").exists():
            return path
    raise FileNotFoundError("Не найден data_registry/dataset_candidate_registry.csv")

PROJECT_ROOT = find_project_root()
REGISTRY_PATH = PROJECT_ROOT / "data_registry" / "dataset_candidate_registry.csv"

UCI_SUMMARY_PATH = PROJECT_ROOT / "data_registry" / "uci_data_preview_summary.csv"
UCI_FEATURES_PATH = PROJECT_ROOT / "data_registry" / "uci_feature_preview.csv"
UCI_ERRORS_PATH = PROJECT_ROOT / "data_registry" / "uci_fetch_errors.csv"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("REGISTRY_PATH =", REGISTRY_PATH)
print("UCI_SUMMARY_PATH =", UCI_SUMMARY_PATH)
print("UCI_FEATURES_PATH =", UCI_FEATURES_PATH)
print("UCI_ERRORS_PATH =", UCI_ERRORS_PATH)

PROJECT_ROOT = C:\Users\Vanargo\Desktop\ML-CRA
REGISTRY_PATH = C:\Users\Vanargo\Desktop\ML-CRA\data_registry\dataset_candidate_registry.csv
UCI_SUMMARY_PATH = C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_data_preview_summary.csv
UCI_FEATURES_PATH = C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_feature_preview.csv
UCI_ERRORS_PATH = C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_fetch_errors.csv


In [12]:
if importlib.util.find_spec("ucimlrepo") is None:
    print("Пакет ucimlrepo не установлен. Выполни в активном окружении:")
    print("pip install ucimlrepo")
    raise SystemExit("Установи ucimlrepo и перезапусти тетрадь.")

from ucimlrepo import fetch_ucirepo

print("Пакет ucimlrepo установлен.")

Пакет ucimlrepo установлен.


In [13]:
registry = pd.read_csv(REGISTRY_PATH, dtype=str, keep_default_na=False)

uci_candidate_configs = [
    {
        "candidate_id": "uci_online_shoppers",
        "uci_id": 468,
        "dataset_name": "Online Shoppers Purchasing Intention",
        "task_type": "classification",
        "target_candidates": ["Revenue"],
    },
    {
        "candidate_id": "uci_appliances_energy",
        "uci_id": 374,
        "dataset_name": "Appliances Energy Prediction",
        "task_type": "regression",
        "target_candidates": ["Appliances"],
    },
    {
        "candidate_id": "uci_metro_traffic",
        "uci_id": 492,
        "dataset_name": "Metro Interstate Traffic Volume",
        "task_type": "regression",
        "target_candidates": ["traffic_volume"],
    },
    {
        "candidate_id": "uci_bank_marketing",
        "uci_id": 222,
        "dataset_name": "Bank Marketing",
        "task_type": "classification",
        "target_candidates": ["y"],
    },
    {
        "candidate_id": "uci_cdc_diabetes",
        "uci_id": 891,
        "dataset_name": "CDC Diabetes Health Indicators",
        "task_type": "classification",
        "target_candidates": ["Diabetes_binary", "Diabetes_012", "Diabetes_binary_or_multiclass", "Diabetes"],
    },
]

candidate_ids = [item["candidate_id"] for item in uci_candidate_configs]
registry_subset = registry[registry["candidate_id"].isin(candidate_ids)].copy()

missing_registry_rows = sorted(set(candidate_ids) - set(registry_subset["candidate_id"]))
if missing_registry_rows:
    raise ValueError(f"В реестре отсутствуют UCI-кандидаты: {missing_registry_rows}")

display(registry_subset[[
    "candidate_id",
    "source_name",
    "source_dataset_id",
    "dataset_name",
    "task_type",
    "target_name",
    "status",
]])

,candidate_id,source_name,source_dataset_id,dataset_name,task_type,target_name,status
3,uci_online_shoppers,UCI,,Online Shoppers Purchasing Intention,classification,Revenue,candidate
4,uci_appliances_energy,UCI,,Appliances Energy Prediction,regression,Appliances,candidate
5,uci_metro_traffic,UCI,,Metro Interstate Traffic Volume,regression,traffic_volume,candidate
6,uci_bank_marketing,UCI,222,Bank Marketing,classification,y,conditional
7,uci_cdc_diabetes,UCI,891,CDC Diabetes Health Indicators,classification,Diabetes_binary_or_multiclass,conditional


In [14]:
TIME_PATTERNS = re.compile(
    r"(date|time|timestamp|year|month|day|hour|minute|period|week|weekday)",
    re.IGNORECASE,
)

ID_PATTERNS = re.compile(
    r"(^id$|^id_|_id$|identifier|user|customer|client|session|visitor|account|row_id|hash)",
    re.IGNORECASE,
)

def safe_dtype_group(series: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    if isinstance(series.dtype, pd.CategoricalDtype):
        return "categorical"
    return "object_or_string"

def high_cardinality_flag(series: pd.Series) -> str:
    n = len(series)
    nunique = series.nunique(dropna=True)
    dtype_group = safe_dtype_group(series)
    feature_name = str(series.name)

    if n == 0:
        return "no"

    if dtype_group == "numeric":
        if ID_PATTERNS.search(feature_name) and nunique >= 50:
            return "possible_numeric_identifier"
        return "not_applicable_numeric"

    if nunique >= 50 and nunique / n >= 0.01:
        return "yes"

    return "no"

def select_target_column(y_raw: pd.DataFrame | pd.Series, target_candidates: list[str]) -> tuple[pd.Series, str, str]:
    if isinstance(y_raw, pd.Series):
        target_name = y_raw.name or "target"
        return y_raw.rename(target_name), target_name, "series"

    if not isinstance(y_raw, pd.DataFrame):
        y_series = pd.Series(y_raw, name="target")
        return y_series, "target", "array_like"

    if y_raw.shape[1] == 0:
        raise ValueError("В data.targets нет столбцов.")

    for target_candidate in target_candidates:
        if target_candidate in y_raw.columns:
            return y_raw[target_candidate].rename(target_candidate), target_candidate, "matched_candidate"

    if y_raw.shape[1] == 1:
        target_name = str(y_raw.columns[0])
        return y_raw.iloc[:, 0].rename(target_name), target_name, "single_column_fallback"

    target_name = str(y_raw.columns[0])
    return y_raw.iloc[:, 0].rename(target_name), target_name, "first_column_fallback_multiple_targets"

def summarize_classification_target(y: pd.Series) -> tuple[str, str]:
    counts = y.value_counts(dropna=False)
    proportions = (counts / len(y)).round(6)

    count_repr = "; ".join([f"{idx}={val}" for idx, val in counts.items()])
    prop_repr = "; ".join([f"{idx}={val}" for idx, val in proportions.items()])

    return count_repr, prop_repr

def summarize_regression_target(y: pd.Series) -> tuple[str, str]:
    numeric_y = pd.to_numeric(y, errors="coerce")
    description = numeric_y.describe(percentiles=[0.25, 0.5, 0.75])

    summary_items = {
        "count": description.get("count", ""),
        "mean": description.get("mean", ""),
        "std": description.get("std", ""),
        "min": description.get("min", ""),
        "q25": description.get("25%", ""),
        "median": description.get("50%", ""),
        "q75": description.get("75%", ""),
        "max": description.get("max", ""),
    }

    summary_repr = "; ".join([
        f"{key}={round(float(value), 6)}"
        for key, value in summary_items.items()
        if value != "" and pd.notna(value)
    ])

    missing_repr = f"missing={int(numeric_y.isna().sum())}"

    return summary_repr, missing_repr

def sample_values(series: pd.Series, max_values: int = 5) -> str:
    values = series.dropna().head(max_values).tolist()
    return "; ".join([str(value) for value in values])

In [15]:
summary_rows: list[dict[str, Any]] = []
feature_rows: list[dict[str, Any]] = []
error_rows: list[dict[str, Any]] = []

for config in uci_candidate_configs:
    candidate_id = config["candidate_id"]
    uci_id = config["uci_id"]
    dataset_name = config["dataset_name"]
    task_type = config["task_type"]
    target_candidates = config["target_candidates"]

    print("\n" + "=" * 100)
    print(f"Кандидат: {candidate_id} | UCI ID: {uci_id}")

    try:
        dataset = fetch_ucirepo(id=uci_id)

        X = dataset.data.features
        y_raw = dataset.data.targets

        if X is None:
            raise ValueError("dataset.data.features is None")
        if y_raw is None:
            raise ValueError("dataset.data.targets is None")

        X = pd.DataFrame(X).copy()
        y, selected_target_name, target_selection_method = select_target_column(
            y_raw=y_raw,
            target_candidates=target_candidates,
        )

        if task_type == "classification":
            target_summary, target_secondary_summary = summarize_classification_target(y)
        else:
            target_summary, target_secondary_summary = summarize_regression_target(y)

        missing_total = int(X.isna().sum().sum())
        missing_columns = [col for col in X.columns if X[col].isna().sum() > 0]

        time_candidates = [col for col in X.columns if TIME_PATTERNS.search(str(col))]
        id_candidates = [col for col in X.columns if ID_PATTERNS.search(str(col))]

        dtype_counts = X.apply(safe_dtype_group).value_counts().to_dict()

        duration_feature_present = "yes" if "duration" in [str(col) for col in X.columns] else "no"

        summary_rows.append({
            "candidate_id": candidate_id,
            "uci_id": uci_id,
            "dataset_name": dataset_name,
            "task_type": task_type,
            "target_name": selected_target_name,
            "target_selection_method": target_selection_method,
            "n_rows_loaded": len(X),
            "n_features_loaded": X.shape[1],
            "target_missing_count": int(y.isna().sum()),
            "target_summary": target_summary,
            "target_secondary_summary": target_secondary_summary,
            "feature_missing_total": missing_total,
            "feature_missing_columns_count": len(missing_columns),
            "time_column_candidates": "; ".join(time_candidates),
            "id_column_candidates": "; ".join(id_candidates),
            "duration_feature_present": duration_feature_present,
            "dtype_counts": str(dtype_counts),
            "memory_usage_mb": round(float(X.memory_usage(deep=True).sum()) / 1024 / 1024, 3),
            "fetch_status": "success",
        })

        for col in X.columns:
            s = X[col]
            feature_rows.append({
                "candidate_id": candidate_id,
                "uci_id": uci_id,
                "feature_name": col,
                "dtype": str(s.dtype),
                "dtype_group": safe_dtype_group(s),
                "missing_count": int(s.isna().sum()),
                "missing_share": round(float(s.isna().mean()), 6),
                "n_unique": int(s.nunique(dropna=True)),
                "high_cardinality_flag": high_cardinality_flag(s),
                "time_name_candidate": "yes" if TIME_PATTERNS.search(str(col)) else "no",
                "id_name_candidate": "yes" if ID_PATTERNS.search(str(col)) else "no",
                "sample_values": sample_values(s),
            })

        print("Загрузка успешна.")
        print("Размер X:", X.shape)
        print("Размер y:", y.shape)
        print("Целевая переменная:", selected_target_name)
        print("Метод выбора целевой переменной:", target_selection_method)
        print("Типы признаков:", dtype_counts)
        print("Пропуски в признаках:", missing_total)
        print("Временные признаки-кандидаты:", time_candidates)
        print("Идентификаторные признаки-кандидаты:", id_candidates)
        print("duration присутствует:", duration_feature_present)

        if task_type == "classification":
            display(y.value_counts(dropna=False).to_frame("count").assign(share=lambda df: df["count"] / len(y)))
        else:
            display(pd.to_numeric(y, errors="coerce").describe().to_frame("target_description"))

        display(X.head())

    except Exception as exc:
        error_rows.append({
            "candidate_id": candidate_id,
            "uci_id": uci_id,
            "dataset_name": dataset_name,
            "error_type": type(exc).__name__,
            "error_message": repr(exc),
            "decision_needed": "manual_review_required",
        })

        print("Ошибка загрузки или просмотра.")
        print(type(exc).__name__, repr(exc))


Кандидат: uci_online_shoppers | UCI ID: 468
Ошибка загрузки или просмотра.
ConnectionError ConnectionError('Error connecting to server')

Кандидат: uci_appliances_energy | UCI ID: 374
Ошибка загрузки или просмотра.
ConnectionError ConnectionError('Error connecting to server')

Кандидат: uci_metro_traffic | UCI ID: 492
Ошибка загрузки или просмотра.
ConnectionError ConnectionError('Error connecting to server')

Кандидат: uci_bank_marketing | UCI ID: 222
Ошибка загрузки или просмотра.
ConnectionError ConnectionError('Error connecting to server')

Кандидат: uci_cdc_diabetes | UCI ID: 891
Ошибка загрузки или просмотра.
ConnectionError ConnectionError('Error connecting to server')


In [16]:
summary_columns = [
    "candidate_id",
    "uci_id",
    "dataset_name",
    "task_type",
    "target_name",
    "target_selection_method",
    "n_rows_loaded",
    "n_features_loaded",
    "target_missing_count",
    "target_summary",
    "target_secondary_summary",
    "feature_missing_total",
    "feature_missing_columns_count",
    "time_column_candidates",
    "id_column_candidates",
    "duration_feature_present",
    "dtype_counts",
    "memory_usage_mb",
    "fetch_status",
]

feature_columns = [
    "candidate_id",
    "uci_id",
    "feature_name",
    "dtype",
    "dtype_group",
    "missing_count",
    "missing_share",
    "n_unique",
    "high_cardinality_flag",
    "time_name_candidate",
    "id_name_candidate",
    "sample_values",
]

error_columns = [
    "candidate_id",
    "uci_id",
    "dataset_name",
    "error_type",
    "error_message",
    "decision_needed",
]

summary_df = pd.DataFrame(summary_rows, columns=summary_columns)
features_df = pd.DataFrame(feature_rows, columns=feature_columns)
errors_df = pd.DataFrame(error_rows, columns=error_columns)

summary_df.to_csv(UCI_SUMMARY_PATH, index=False, encoding="utf-8-sig")
features_df.to_csv(UCI_FEATURES_PATH, index=False, encoding="utf-8-sig")
errors_df.to_csv(UCI_ERRORS_PATH, index=False, encoding="utf-8-sig")

print("Сводка UCI сохранена:", UCI_SUMMARY_PATH)
print("Сводка признаков UCI сохранена:", UCI_FEATURES_PATH)
print("Журнал ошибок UCI сохранен:", UCI_ERRORS_PATH)

print("\nСводка:")
display(summary_df)

print("\nОшибки загрузки:")
display(errors_df)

Сводка UCI сохранена: C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_data_preview_summary.csv
Сводка признаков UCI сохранена: C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_feature_preview.csv
Журнал ошибок UCI сохранен: C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_fetch_errors.csv

Сводка:


,candidate_id,uci_id,dataset_name,task_type,target_name,target_selection_method,n_rows_loaded,n_features_loaded,target_missing_count,target_summary,target_secondary_summary,feature_missing_total,feature_missing_columns_count,time_column_candidates,id_column_candidates,duration_feature_present,dtype_counts,memory_usage_mb,fetch_status



Ошибки загрузки:


,candidate_id,uci_id,dataset_name,error_type,error_message,decision_needed
0,uci_online_shoppers,468,Online Shoppers Purchasing Intention,ConnectionError,ConnectionError('Error connecting to server'),manual_review_required
1,uci_appliances_energy,374,Appliances Energy Prediction,ConnectionError,ConnectionError('Error connecting to server'),manual_review_required
2,uci_metro_traffic,492,Metro Interstate Traffic Volume,ConnectionError,ConnectionError('Error connecting to server'),manual_review_required
3,uci_bank_marketing,222,Bank Marketing,ConnectionError,ConnectionError('Error connecting to server'),manual_review_required
4,uci_cdc_diabetes,891,CDC Diabetes Health Indicators,ConnectionError,ConnectionError('Error connecting to server'),manual_review_required


In [17]:
print("Контроль: эта тетрадь не изменяет реестр и не меняет статусы кандидатов.")

status_view = registry[registry["candidate_id"].isin(candidate_ids)][[
    "candidate_id",
    "status",
    "decision_reason",
]]

display(status_view)

accepted = registry[registry["status"].str.lower().eq("accepted")]
if len(accepted) > 0:
    print("Внимание: в реестре уже есть accepted-статусы. Эта тетрадь их не создавала.")
else:
    print("В реестре нет accepted-статусов.")

Контроль: эта тетрадь не изменяет реестр и не меняет статусы кандидатов.


,candidate_id,status,decision_reason
3,uci_online_shoppers,candidate,Good candidate for metric-conflict and class-i...
4,uci_appliances_energy,candidate,Good temporal regression candidate; regression...
5,uci_metro_traffic,candidate,Good temporal regression candidate; risk of dr...
6,uci_bank_marketing,conditional,Useful but overused; duration feature requires...
7,uci_cdc_diabetes,conditional,"No principled objection from John, but medical..."


В реестре нет accepted-статусов.


In [18]:
print("Итоговые файлы для отправки в чат:")
print(UCI_SUMMARY_PATH)
print(UCI_FEATURES_PATH)
print(UCI_ERRORS_PATH)

Итоговые файлы для отправки в чат:
C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_data_preview_summary.csv
C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_feature_preview.csv
C:\Users\Vanargo\Desktop\ML-CRA\data_registry\uci_fetch_errors.csv
